In [33]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import ee
import geemap
import time

In [35]:
plt.rcParams['font.family'] = 'serif'

In [36]:
ee.Authenticate()
ee.Initialize(project = 'ajoyiirs')

In [37]:
path = r'/content/drive/MyDrive/Track_Satellite'
file = os.path.join(path, 'points.csv')
df = pd.read_csv(file)
print("Columns in your CSV:", df.columns.tolist())
print(df.head())

Columns in your CSV: ['ID', 'Name', 'Lat', 'Long']
   ID Name      Lat     Long
0   0    A  28.7054  76.7439
1   1    B  25.0760  73.7969
2   2    C  20.4366  84.6415
3   3    D  23.3823  87.8769
4   4    E  32.4813  75.7263


In [38]:
# Satellite options

ALL_SATELLITES = [
    ("L7",  ee.ImageCollection('LANDSAT/LE07/C02/T1_L2'), "1999-01-01", "SR_B1", True),
    ("L8",  ee.ImageCollection('LANDSAT/LC08/C02/T1_L2'), "2013-04-01", "SR_B2", True),
    ("L9",  ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'), "2022-01-01", "SR_B2", True),
    ("S2",  ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED'), "2017-03-28", "B2", True),
    ("Ast", ee.ImageCollection("ASTER/AST_L1T_003"),       "2000-03-06", "B01", False),
]

SATELLITES = [(label, col, start, band) for label, col, start, band, use in ALL_SATELLITES if use]
print("Satellites selected:", [s[0] for s in SATELLITES])

Satellites selected: ['L7', 'L8', 'L9', 'S2']


In [39]:
from datetime import datetime, UTC

TODAY = datetime.now(UTC).strftime("%Y-%m-%d")

In [40]:
def get_lat_lon(row):
    """Find latitude and longitude columns regardless of how they are named."""
    col_map = {}
    for c in row.index:
        col_map[c.strip().lower()] = c
    lat_key = next((col_map[k] for k in ['latitude','lat'] if k in col_map), None)
    lon_key = next((col_map[k] for k in ['longitude','lon','long'] if k in col_map), None)
    if lat_key and lon_key:
        return float(row[lat_key]), float(row[lon_key])
    raise KeyError(f"Cannot find lat/lon columns. Available: {list(row.index)}")

def fetch_dates_yearly(collection, point, start_year, end_year, band, retries=3):
    """
    Query one year at a time to avoid GEE's 5000-element getInfo() limit.
    Returns a list of timestamps (ms since epoch).
    """
    all_ts = []
    for year in range(start_year, end_year + 1):
        date_start = f"{year}-01-01"
        date_end   = f"{year}-12-31"
        for attempt in range(1, retries + 1):
            try:
                col = (collection
                       .filterBounds(point)
                       .filterDate(date_start, date_end)
                       .select([band]))
                ts_list = col.aggregate_array('system:time_start').getInfo()
                all_ts.extend(ts_list)
                break
            except Exception as e:
                if attempt == retries:
                    print(f"       Year {year} failed after {retries} attempts: {e}")
                else:
                    wait = 5 * attempt
                    print(f"        Year {year} attempt {attempt} failed, retrying in {wait}s...")
                    time.sleep(wait)
    return all_ts

def get_records_for_point_sat(row, sat_label, collection, start_date, band):
    """
    Returns a list of records for one point and one satellite.
    Uses yearly chunking to avoid GEE timeouts.
    """
    lat, lon = get_lat_lon(row)
    point = ee.Geometry.Point([lon, lat])

    start_year = int(start_date[:4])
    end_year   = int(TODAY[:4])

    ts_list = fetch_dates_yearly(collection, point, start_year, end_year, band)

    records = []
    for ts in ts_list:
        date_str = datetime.utcfromtimestamp(ts / 1000).strftime("%d-%m-%Y")
        records.append({
            "ID":        row['ID'],
            "Name":      row['Name'],
            "Date":      date_str,
            "Satellite": sat_label
        })
    return records

print("Helper functions are ready.")


Helper functions are ready.


In [41]:
# Output file — partial results are written here after each point
out_path = os.path.join(path, "satellite_observations.csv")

# Resume support: if the file already exists, load it and skip done combos
if os.path.exists(out_path):
    existing_df = pd.read_csv(out_path)
    done_keys   = set(zip(existing_df['ID'].astype(str), existing_df['Name'], existing_df['Satellite']))
    all_records = existing_df.to_dict('records')
    print(f"  Resuming — {len(existing_df)} rows already saved.")
else:
    done_keys   = set()
    all_records = []
    print("  Starting fresh.")

total_points = len(df)
total_combos = total_points * len(SATELLITES)
combo_done   = 0

for pt_idx, row in df.iterrows():
    pt_label = f"Point '{row['Name']}' (ID={row['ID']})  [{pt_idx+1}/{total_points}]"
    print(f"\n{''*60}")
    print(f"  {pt_label}")
    print(f"{''*60}")

    pt_records = []  # collect all satellites for this point first

    for sat_label, collection, start_date, band in SATELLITES:
        combo_done += 1
        key = (str(row['ID']), row['Name'], sat_label)

        if key in done_keys:
            print(f"  [{combo_done}/{total_combos}]  {sat_label:4s}  — already done, skipping.")
            continue

        print(f"  [{combo_done}/{total_combos}]  {sat_label:4s}  querying {start_date[:4]}–{TODAY[:4]} ...", end=" ", flush=True)

        try:
            recs = get_records_for_point_sat(row, sat_label, collection, start_date, band)
            pt_records.extend(recs)
            print(f"{len(recs):5d} images")
        except Exception as e:
            print(f"\n   ERROR: {e}")

        time.sleep(1)   # polite pause between GEE calls

    # Save after every point (append to master list, write CSV)
    if pt_records:
        all_records.extend(pt_records)
        temp_df = pd.DataFrame(all_records)
        temp_df['_dt'] = pd.to_datetime(temp_df['Date'], format='%d-%m-%Y')
        temp_df = temp_df.sort_values('_dt').drop(columns=['_dt']).reset_index(drop=True)
        temp_df.to_csv(out_path, index=False)
        print(f"\n   Progress saved — {len(temp_df)} total rows in {out_path}")

print(f"\n{''*60}")
print(f"  ALL DONE — {len(all_records)} total records collected.")


  Resuming — 8778 rows already saved.


  Point 'A' (ID=0)  [1/5]

  [1/20]  L7    — already done, skipping.
  [2/20]  L8    — already done, skipping.
  [3/20]  L9    — already done, skipping.
  [4/20]  S2    — already done, skipping.


  Point 'B' (ID=1)  [2/5]

  [5/20]  L7    — already done, skipping.
  [6/20]  L8    — already done, skipping.
  [7/20]  L9    — already done, skipping.
  [8/20]  S2    — already done, skipping.


  Point 'C' (ID=2)  [3/5]

  [9/20]  L7    — already done, skipping.
  [10/20]  L8    — already done, skipping.
  [11/20]  L9    — already done, skipping.
  [12/20]  S2    — already done, skipping.


  Point 'D' (ID=3)  [4/5]

  [13/20]  L7    — already done, skipping.
  [14/20]  L8    — already done, skipping.
  [15/20]  L9    — already done, skipping.
  [16/20]  S2    — already done, skipping.


  Point 'E' (ID=4)  [5/5]

  [17/20]  L7    — already done, skipping.
  [18/20]  L8    — already done, skipping.
  [19/20]  L9    — already done, skipping.
  [20/20]

In [42]:
# Load and re-sort the saved CSV

result_df = pd.read_csv(out_path)
result_df['_dt'] = pd.to_datetime(result_df['Date'], format='%d-%m-%Y')
result_df = result_df.sort_values('_dt').drop(columns=['_dt']).reset_index(drop=True)
result_df.to_csv(out_path, index=False)

print(f"Final CSV: {out_path}")
print(f"    Shape   : {result_df.shape[0]} rows  {result_df.shape[1]} columns")
print(f"    Date range: {result_df['Date'].iloc[0]}    {result_df['Date'].iloc[-1]}")
print(f"\nSatellite breakdown:")
print(result_df['Satellite'].value_counts().to_string())
print(f"\nPoint breakdown:")
print(result_df['Name'].value_counts().to_string())
print(f"\nFirst 10 rows:")
result_df.head(10)


Final CSV: /content/drive/MyDrive/Track_Satellite/satellite_observations.csv
    Shape   : 8778 rows  4 columns
    Date range: 30-06-1999    26-05-2026

Satellite breakdown:
Satellite
S2    4101
L7    2504
L8    1623
L9     550

Point breakdown:
Name
E    2730
C    1915
D    1388
A    1384
B    1361

First 10 rows:


,ID,Name,Date,Satellite
0,1,B,30-06-1999,L7
1,4,E,30-06-1999,L7
2,4,E,30-06-1999,L7
3,2,C,08-07-1999,L7
4,0,A,09-07-1999,L7
5,4,E,16-07-1999,L7
6,4,E,16-07-1999,L7
7,0,A,25-07-1999,L7
8,3,D,02-08-1999,L7
9,1,B,17-08-1999,L7
